# Section 5.3 — The SP Heuristic Against the Extensive Form

Reproduces Tables 4–6 from the thesis.  
Runs **EEV, SP (B-PHA), DE, WS** on 10-cage, 30-cage, and 60-cage instances.

| Instance | DE | Note |
|----------|-----|------|
| 10 cages | tractable | MIP gap 2% |
| 30 cages | tractable (slow) | MIP gap 3% |
| 60 cages | **OOM** | DE skipped |


In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'models'))

import time
import numpy as np
import pandas as pd

from instance import (
    T, loc_mab, regional_mab,
    units_df as full_units_df,
    build_scenarios,
    temps_bad_12, temps_normal_12, temps_good_12,
)
from IP import SalmonFarmingMILP
from DE import DeterministicEquivalent
from SP import AugmentedLagrangianDecomposition


In [2]:
# WS helper

def run_ws_for(units_df_sub, mip_gap=0.02):
    """Wait-and-See: solve each of 81 scenarios independently."""
    scenarios = build_scenarios()
    n_feas = 0
    ws_obj = 0.0
    t0 = time.time()

    for sc_idx, (sc_name, temps_sc, S_sc, prob) in enumerate(scenarios):
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
            scenario_name=sc_name,
        )
        milp.model.Params.OutputFlag = 0
        milp.model.Params.MIPGap = mip_gap
        milp.model.optimize()
        if milp.model.SolCount > 0:
            n_feas += 1
            ws_obj += prob * milp.model.ObjVal

    return ws_obj, time.time() - t0, n_feas


In [3]:
# EEV helper

NA_MONTHS = set(range(0, 45))  # stages 0-2 are fixed


def run_eev_for(units_df_sub, mip_gap=0.02):
    """
    EEV: solve EV (deterministic, expected-parameter) model, fix its stage 0-2
    decisions (stk, harv, h_exist, q), then evaluate across all 81 scenarios.
    Returns (eev_obj, elapsed_s, n_feas_81).
    """
    # Step 1: EV model (expected temperatures, blended survival)
    temps_exp = np.tile(
        np.array([5, 5, 5, 6, 9, 12, 14, 16.5, 15.5, 13, 10, 7.5]),
        (T // 12) + 1
    )[:T]
    S_exp = np.full(T, (2.0 * (1.0 - 0.0002) ** 30 + (1.0 - 0.001) ** 30) / 3.0)

    t0 = time.time()
    ev_m = SalmonFarmingMILP(
        units_df=units_df_sub, temps_t=temps_exp, survival_rates=S_exp,
        horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
        density_limit=25.0, verbose=False,
    )
    ev_m.model.Params.OutputFlag = 0
    ev_m.model.Params.MIPGap = mip_gap
    ev_m.model.optimize()
    if ev_m.model.SolCount == 0:
        print('  EV model infeasible')
        return None, time.time() - t0, 0

    # Step 2: Extract stage 0-2 decisions
    ev_stk = {
        (u, t): round(ev_m.variables['z'][u, t].X)
        for u in ev_m.U for t in ev_m.Tset if t in NA_MONTHS
    }
    ev_harv = {
        (u, s, t): round(ev_m.variables['h'][u, s, t].X)
        for u in ev_m.U for s in ev_m.Tset
        for t in ev_m.H_by_us.get((u, s), []) if t in NA_MONTHS
    }
    ev_hexist = {
        (u, t): round(ev_m.variables['h_exist'][u, t].X)
        for u in ev_m.U_exist for t in ev_m.Tset if t in NA_MONTHS
    }
    ev_q = {
        (u, s): ev_m.variables['q'][u, s].X
        for u in ev_m.U for s in ev_m.Tset
        if s in NA_MONTHS and round(ev_m.variables['z'][u, s].X) == 1
    }

    # Step 3: Fix EV decisions and evaluate on all 81 scenarios
    scenarios = build_scenarios()
    n_feas = 0
    eev_obj = 0.0
    feas_prob = 0.0

    for sc_idx, (sc_name, temps_sc, S_sc, prob) in enumerate(scenarios):
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
            scenario_name=sc_name, disable_economic_presolve=True,
        )
        model = milp.model
        model.update()

        for (u, t), val in ev_stk.items():
            v = model.getVarByName(f'z[{u},{t}]')
            if v is not None:
                v.LB = val
                v.UB = val
        for (u, s, t), val in ev_harv.items():
            v = model.getVarByName(f'h[{u},{s},{t}]')
            if v is not None:
                v.LB = val
                v.UB = val
        for (u, t), val in ev_hexist.items():
            v = model.getVarByName(f'h_exist[{u},{t}]')
            if v is not None:
                v.LB = val
                v.UB = val
        for (u, s), val in ev_q.items():
            v = model.getVarByName(f'q[{u},{s}]')
            if v is not None:
                v.LB = val
                v.UB = val

        model.Params.OutputFlag = 0
        model.Params.MIPGap = mip_gap
        model.update()
        model.optimize()

        if model.SolCount > 0:
            n_feas += 1
            feas_prob += prob
            eev_obj += prob * model.ObjVal

    eev_normalized = eev_obj / feas_prob if feas_prob > 0 else 0.0
    return eev_normalized, time.time() - t0, n_feas


In [4]:
# Main experiment: {EEV, SP, DE, WS} x {10, 30, 60 cages}
# DE is attempted for all sizes but expected to OOM at 60 cages.
# Warning: 30-cage DE can take several hours.

import gurobipy as gb

results = []

In [5]:
# 10 cages
n_cages = 10
mip_gap = 0.02
sub_df  = full_units_df.head(n_cages).reset_index(drop=True)
print(f'\n{"="*70}')
print(f'  n_cages={n_cages}   mip_gap={mip_gap:.0%}')
print(f'{"="*70}')

print(f'\n--- EEV ({n_cages} cages) ---')
eev_obj, eev_time, eev_feas = run_eev_for(sub_df, mip_gap=mip_gap)
print(f'  EEV = {eev_obj/1e6:.1f} MNOK  |  {eev_time:.1f}s  |  {eev_feas}/81 feasible')

print(f'\n--- SP ({n_cages} cages) ---')
ald = AugmentedLagrangianDecomposition(
    units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
    T=T, mip_gap=mip_gap,
    temps_bad=temps_bad_12, temps_normal=temps_normal_12, temps_good=temps_good_12,
)
ald.build()
ald.solve()
sp_obj  = ald.eval_obj
sp_time = ald.total_time
print(f'  SP  = {sp_obj/1e6:.1f} MNOK  |  {sp_time:.1f}s  |  {ald.n_feasible}/{ald.n_scenarios} feasible')
del ald

print(f'\n--- DE ({n_cages} cages) ---')
de_obj = de_time = de_gap = None
try:
    de = DeterministicEquivalent(
        units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
        T=T, mip_gap=mip_gap,
    )
    de.build()
    de.solve()
    de_obj  = de.obj_val
    de_time = de.solve_time
    de_gap  = de.model.MIPGap if de.model.SolCount > 0 else float('nan')
    print(f'  DE  = {de_obj/1e6:.1f} MNOK  |  {de_time:.1f}s  |  gap {de_gap:.2%}')
except (MemoryError, gb.GurobiError) as exc:
    print(f'  DE  = OOM ({exc})')
finally:
    try:
        de.model.dispose()
    except Exception:
        pass
    try:
        del de
    except Exception:
        pass

print(f'\n--- WS ({n_cages} cages) ---')
ws_obj, ws_time, ws_feas = run_ws_for(sub_df, mip_gap=mip_gap)
print(f'  WS  = {ws_obj/1e6:.1f} MNOK  |  {ws_time:.1f}s  |  {ws_feas}/81 feasible')

vss_sp  = (sp_obj  - eev_obj) / 1e6
vss_de  = (de_obj  - eev_obj) / 1e6 if de_obj is not None else None
evpi_sp = (ws_obj  - sp_obj)  / 1e6
evpi_de = (ws_obj  - de_obj)  / 1e6 if de_obj is not None else None
print(f'  VSS_SP={vss_sp:.1f}  VSS_DE={vss_de}  EVPI_SP={evpi_sp:.1f}  EVPI_DE={evpi_de}')

results.append({
    'n_cages':        n_cages,
    'EEV [MNOK]':     round(eev_obj / 1e6, 1),
    'EEV feas/81':    eev_feas,
    'SP [MNOK]':      round(sp_obj  / 1e6, 1),
    'DE [MNOK]':      round(de_obj  / 1e6, 1) if de_obj is not None else 'OOM',
    'WS [MNOK]':      round(ws_obj  / 1e6, 1),
    'VSS_SP [MNOK]':  round(vss_sp,  1),
    'VSS_DE [MNOK]':  round(vss_de,  1) if vss_de is not None else 'N/A',
    'EVPI_SP [MNOK]': round(evpi_sp, 1),
    'EVPI_DE [MNOK]': round(evpi_de, 1) if evpi_de is not None else 'N/A',
    'EEV time [s]':   round(eev_time,  1),
    'SP time [s]':    round(sp_time,   1),
    'DE time [s]':    round(de_time,   1) if de_time is not None else 'OOM',
    'WS time [s]':    round(ws_time,   1),
    'MIP gap':        mip_gap,
})


  n_cages=10   mip_gap=2%

--- EEV (10 cages) ---
Set parameter Username
Set parameter LicenseID to value 2786519
Academic license - for non-commercial use only - expires 2027-03-03
  EEV = 627.4 MNOK  |  54.3s  |  54/81 feasible

--- SP (10 cages) ---
Built 81 scenarios
NA variables: 91180 total (91180 binary)
Variable cache built (tree-aware)


Step 0 (initial solve): 100%|██████████| 81/81 [00:43<00:00,  1.85sc/s]


Step 0 done — E[obj]: 788,227,315
Auto-calibrated rho_bin=7.108e+06, rho_max_bin=7.882e+09

Step 0 | Obj: 788227314.56 | MaxDev: 9.63e-01 | AvgBin: 0.0005 | Fixed: 0
Starting PH iterations (max 1000)
  [Iter 1] progress: 5/81 done, 76 running
  [Iter 1] progress: 10/81 done, 71 running
  [Iter 1] progress: 15/81 done, 66 running
  [Iter 1] heartbeat: 16/81 done, 65 running, oldest=5.3s | pending: 04,05,16,19,20,21,22,23,24,25,26,27,...
  [Iter 1] progress: 20/81 done, 61 running
  [Iter 1] progress: 25/81 done, 56 running
  [Iter 1] progress: 30/81 done, 51 running
  [Iter 1] progress: 35/81 done, 46 running
  [Iter 1] progress: 40/81 done, 41 running
  [Iter 1] heartbeat: 40/81 done, 41 running, oldest=10.4s | pending: 24,41,42,43,44,45,46,47,48,49,50,51,...
  [Iter 1] progress: 45/81 done, 36 running
  [Iter 1] progress: 50/81 done, 31 running
  [Iter 1] progress: 55/81 done, 26 running
  [Iter 1] progress: 60/81 done, 21 running
  [Iter 1] heartbeat: 63/81 done, 18 running, oldest=1

In [ ]:
# 30 cages — Warning: DE can take several hours
n_cages = 30
mip_gap = 0.03
sub_df  = full_units_df.head(n_cages).reset_index(drop=True)
print(f'\n{"="*70}')
print(f'  n_cages={n_cages}   mip_gap={mip_gap:.0%}')
print(f'{"="*70}')

print(f'\n--- EEV ({n_cages} cages) ---')
eev_obj, eev_time, eev_feas = run_eev_for(sub_df, mip_gap=mip_gap)
print(f'  EEV = {eev_obj/1e6:.1f} MNOK  |  {eev_time:.1f}s  |  {eev_feas}/81 feasible')

print(f'\n--- SP ({n_cages} cages) ---')
ald = AugmentedLagrangianDecomposition(
    units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
    T=T, mip_gap=mip_gap,
    temps_bad=temps_bad_12, temps_normal=temps_normal_12, temps_good=temps_good_12,
)
ald.build()
ald.solve()
sp_obj  = ald.eval_obj
sp_time = ald.total_time
print(f'  SP  = {sp_obj/1e6:.1f} MNOK  |  {sp_time:.1f}s  |  {ald.n_feasible}/{ald.n_scenarios} feasible')
del ald

print(f'\n--- DE ({n_cages} cages) ---')
de_obj = de_time = de_gap = None
try:
    de = DeterministicEquivalent(
        units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
        T=T, mip_gap=mip_gap,
    )
    de.build()
    de.solve()
    de_obj  = de.obj_val
    de_time = de.solve_time
    de_gap  = de.model.MIPGap if de.model.SolCount > 0 else float('nan')
    print(f'  DE  = {de_obj/1e6:.1f} MNOK  |  {de_time:.1f}s  |  gap {de_gap:.2%}')
except (MemoryError, gb.GurobiError) as exc:
    print(f'  DE  = OOM ({exc})')
finally:
    try:
        de.model.dispose()
    except Exception:
        pass
    try:
        del de
    except Exception:
        pass

print(f'\n--- WS ({n_cages} cages) ---')
ws_obj, ws_time, ws_feas = run_ws_for(sub_df, mip_gap=mip_gap)
print(f'  WS  = {ws_obj/1e6:.1f} MNOK  |  {ws_time:.1f}s  |  {ws_feas}/81 feasible')

vss_sp  = (sp_obj  - eev_obj) / 1e6
vss_de  = (de_obj  - eev_obj) / 1e6 if de_obj is not None else None
evpi_sp = (ws_obj  - sp_obj)  / 1e6
evpi_de = (ws_obj  - de_obj)  / 1e6 if de_obj is not None else None
print(f'  VSS_SP={vss_sp:.1f}  VSS_DE={vss_de}  EVPI_SP={evpi_sp:.1f}  EVPI_DE={evpi_de}')

results.append({
    'n_cages':        n_cages,
    'EEV [MNOK]':     round(eev_obj / 1e6, 1),
    'EEV feas/81':    eev_feas,
    'SP [MNOK]':      round(sp_obj  / 1e6, 1),
    'DE [MNOK]':      round(de_obj  / 1e6, 1) if de_obj is not None else 'OOM',
    'WS [MNOK]':      round(ws_obj  / 1e6, 1),
    'VSS_SP [MNOK]':  round(vss_sp,  1),
    'VSS_DE [MNOK]':  round(vss_de,  1) if vss_de is not None else 'N/A',
    'EVPI_SP [MNOK]': round(evpi_sp, 1),
    'EVPI_DE [MNOK]': round(evpi_de, 1) if evpi_de is not None else 'N/A',
    'EEV time [s]':   round(eev_time,  1),
    'SP time [s]':    round(sp_time,   1),
    'DE time [s]':    round(de_time,   1) if de_time is not None else 'OOM',
    'WS time [s]':    round(ws_time,   1),
    'MIP gap':        mip_gap,
})

In [ ]:
# 60 cages — DE expected to OOM
n_cages = 60
mip_gap = 0.02
sub_df  = full_units_df.head(n_cages).reset_index(drop=True)
print(f'\n{"="*70}')
print(f'  n_cages={n_cages}   mip_gap={mip_gap:.0%}')
print(f'{"="*70}')

print(f'\n--- EEV ({n_cages} cages) ---')
eev_obj, eev_time, eev_feas = run_eev_for(sub_df, mip_gap=mip_gap)
print(f'  EEV = {eev_obj/1e6:.1f} MNOK  |  {eev_time:.1f}s  |  {eev_feas}/81 feasible')

print(f'\n--- SP ({n_cages} cages) ---')
ald = AugmentedLagrangianDecomposition(
    units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
    T=T, mip_gap=mip_gap,
    temps_bad=temps_bad_12, temps_normal=temps_normal_12, temps_good=temps_good_12,
)
ald.build()
ald.solve()
sp_obj  = ald.eval_obj
sp_time = ald.total_time
print(f'  SP  = {sp_obj/1e6:.1f} MNOK  |  {sp_time:.1f}s  |  {ald.n_feasible}/{ald.n_scenarios} feasible')
del ald

print(f'\n--- DE ({n_cages} cages) ---')
de_obj = de_time = de_gap = None
try:
    de = DeterministicEquivalent(
        units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
        T=T, mip_gap=mip_gap,
    )
    de.build()
    de.solve()
    de_obj  = de.obj_val
    de_time = de.solve_time
    de_gap  = de.model.MIPGap if de.model.SolCount > 0 else float('nan')
    print(f'  DE  = {de_obj/1e6:.1f} MNOK  |  {de_time:.1f}s  |  gap {de_gap:.2%}')
except (MemoryError, gb.GurobiError) as exc:
    print(f'  DE  = OOM ({exc})')
finally:
    try:
        de.model.dispose()
    except Exception:
        pass
    try:
        del de
    except Exception:
        pass

print(f'\n--- WS ({n_cages} cages) ---')
ws_obj, ws_time, ws_feas = run_ws_for(sub_df, mip_gap=mip_gap)
print(f'  WS  = {ws_obj/1e6:.1f} MNOK  |  {ws_time:.1f}s  |  {ws_feas}/81 feasible')

vss_sp  = (sp_obj  - eev_obj) / 1e6
vss_de  = (de_obj  - eev_obj) / 1e6 if de_obj is not None else None
evpi_sp = (ws_obj  - sp_obj)  / 1e6
evpi_de = (ws_obj  - de_obj)  / 1e6 if de_obj is not None else None
print(f'  VSS_SP={vss_sp:.1f}  VSS_DE={vss_de}  EVPI_SP={evpi_sp:.1f}  EVPI_DE={evpi_de}')

results.append({
    'n_cages':        n_cages,
    'EEV [MNOK]':     round(eev_obj / 1e6, 1),
    'EEV feas/81':    eev_feas,
    'SP [MNOK]':      round(sp_obj  / 1e6, 1),
    'DE [MNOK]':      round(de_obj  / 1e6, 1) if de_obj is not None else 'OOM',
    'WS [MNOK]':      round(ws_obj  / 1e6, 1),
    'VSS_SP [MNOK]':  round(vss_sp,  1),
    'VSS_DE [MNOK]':  round(vss_de,  1) if vss_de is not None else 'N/A',
    'EVPI_SP [MNOK]': round(evpi_sp, 1),
    'EVPI_DE [MNOK]': round(evpi_de, 1) if evpi_de is not None else 'N/A',
    'EEV time [s]':   round(eev_time,  1),
    'SP time [s]':    round(sp_time,   1),
    'DE time [s]':    round(de_time,   1) if de_time is not None else 'OOM',
    'WS time [s]':    round(ws_time,   1),
    'MIP gap':        mip_gap,
})

In [6]:
# Results table

df_results = pd.DataFrame(results).set_index('n_cages')
display(df_results)


,EEV [MNOK],EEV feas/81,SP [MNOK],DE [MNOK],WS [MNOK],VSS_SP [MNOK],VSS_DE [MNOK],EVPI_SP [MNOK],EVPI_DE [MNOK],EEV time [s],SP time [s],DE time [s],WS time [s],MIP gap
n_cages,,,,,,,,,,,,,,
10,627.4,54,666.3,691.1,788.7,38.9,63.7,122.4,97.5,54.3,177.5,606.3,203.9,0.02
